In [1]:
pip install pandas torch torchvision transformers pillow tqdm nltk matplotlib seaborn


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Importing necessary libraries

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
import ast
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import io
import random
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import nltk
import re
import torch
import torch.nn as nn
from torchvision import models



# Problem & Dataset

## Loading RSICD Dataset
### Exploring the dataset a bit

In [ ]:
train_csv_path = 'train.csv'
train_df = pd.read_csv(train_csv_path)
print("--- First 5 rows of train.csv ---")
print(train_df)
print("\n--- DataFrame Info ---")
train_df.info()


val_csv_path = 'valid.csv'
val_df = pd.read_csv(val_csv_path)
print("--- First 5 rows of val.csv ---")
print(val_df)
print("\n--- DataFrame Info ---")
val_df.info()

test_csv_path = 'test.csv'
test_df = pd.read_csv(test_csv_path)
print("--- First 5 rows of test.csv ---")
print(test_df)
print("\n--- DataFrame Info ---")
test_df.info()

--- First 5 rows of train.csv ---
                          filename  \
0       rsicd_images/airport_1.jpg   
1      rsicd_images/airport_10.jpg   
2     rsicd_images/airport_100.jpg   
3     rsicd_images/airport_101.jpg   
4     rsicd_images/airport_102.jpg   
...                            ...   
8729        rsicd_images/00914.jpg   
8730        rsicd_images/00915.jpg   
8731        rsicd_images/00916.jpg   
8732        rsicd_images/00918.jpg   
8733        rsicd_images/00920.jpg   

                                               captions  \
0     ['Many aircraft are parked next to a long buil...   
1     ['some planes are parked in an airport.'\n 'th...   
2     ['Many aircraft are parked in an airport near ...   
3     ['Many aircraft are parked near a large buildi...   
4     ['several buildings and green trees are around...   
...                                                 ...   
8729  ['the majestic polygonal baseball field can co...   
8730  ['the baseball field is near th

## Preprocessing


**Why Resize to 224x224**?
#### Fixed Input Size: Convolutional Neural Networks (CNNs), after their convolution layers, typically have fully-connected layers. These final layers expect a feature vector of a fixed size. This means the input image to the network must also be a fixed size.
#### ImageNet Standard: The specific size, 224x224, is the standard input dimension for the vast majority of models (including ResNet and MobileNet that we will use) which were pre-trained on the ImageNet dataset. To use their pre-trained "knowledge" effectively, we must feed our images to them in the exact same format they were originally trained on. Using a different size would cause a mismatch and either throw an error or lead to very poor performance.
---
**Why ImageNet Normalization**?
#### Consistent Data Distribution: Normalization is the process of transforming the pixel values of an image to a standard range. ImageNet pre-trained models are not just used to a specific size, they are also used to a specific distribution of pixel values.
#### For ImageNet, the standard practice is to first scale pixel values from the [0, 255] range to [0.0, 1.0], and then normalize them using the following mean and standard deviation for each of the Red, Green, and Blue channels:
```python
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
```


In [ ]:
def clean_and_split_captions(captions_str):
    """
    Cleans the messy caption string from the CSV.
    1. It's a string representation of a list
    2. The strings inside the list are sometimes concatenated so we are going to split them by period.
    """
    try:
        captions_list = ast.literal_eval(captions_str)

        # Join everything into one big block, then split by period.
        # This handles cases where one list element contains multiple sentences.
        full_text = ' '.join(captions_list)

        # Split by period and clean up whitespace. Filter out any empty strings.
        sentences = [s.strip() for s in full_text.split('.') if s.strip()]

        return sentences
    except (ValueError, SyntaxError):
        return []


print("Cleaning captions for all data splits...")
train_df['captions'] = train_df['captions'].apply(clean_and_split_captions)
val_df['captions'] = val_df['captions'].apply(clean_and_split_captions)
test_df['captions'] = test_df['captions'].apply(clean_and_split_captions)

print("\n--- Verifying the fix on the first row of training data ---")
print(train_df.iloc[0]['captions'])

Cleaning captions for all data splits...

--- Verifying the fix on the first row of training data ---
['Many aircraft are parked next to a long building in an airport', 'Many planes are parked next to a long building at an airport', 'Many planes are parked next to a long building in an airport', 'many planes are parked next to a long building at an airport', 'many planes are parked next to a long building in an airport']


In [ ]:
image_transforms = transforms.Compose([
    # 1. Resize the image to 224x224 pixels.
    transforms.Resize((224, 224)),

    # 2. Convert the image to a PyTorch Tensor. This also scales the pixel
    #    values from a range of [0, 255] to a range of [0.0, 1.0].
    #    It also changes the dimension order from (H, W, C) to (C, H, W).
    transforms.ToTensor(),

    # 3. Normalize the tensor image with the mean and standard deviation 
    # from the ImageNet dataset.
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# --- Let's test it on a sample image --
# Get the first image's byte data from our DataFrame
first_image_bytes = ast.literal_eval(train_df.iloc[0]['image'])['bytes']

# Open the image from bytes
img = Image.open(io.BytesIO(first_image_bytes)).convert("RGB") # Ensure it's RGB

print("Original image size:", img.size)
transformed_img_tensor = image_transforms(img)
print("Transformed tensor shape:", transformed_img_tensor.shape)
print("Transformed tensor dtype:", transformed_img_tensor.dtype)

Original image size: (224, 224)
Transformed tensor shape: torch.Size([3, 224, 224])
Transformed tensor dtype: torch.float32


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/arnavagarwal/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/arnavagarwal/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
class Vocabulary:
    def __init__(self):
        # The order of these special tokens is important
        self.itos = {0: "<pad>", 1: "<bos>", 2: "<eos>", 3: "<unk>"}
        self.stoi = {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3}

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer(text):
        text = text.lower()
        return [tok.strip() for tok in nltk.tokenize.word_tokenize(text)]

    def build_vocabulary(self, captions_series, vocab_size):
        frequencies = Counter()
        idx = 4

        print("Counting word frequencies from clean data...")
        for caption_list in tqdm(captions_series):
            for sentence in caption_list:
                frequencies.update(self.tokenizer(sentence))

        most_common = frequencies.most_common(vocab_size - len(self.itos))

        for word, _ in most_common:
            self.stoi[word] = idx
            self.itos[idx] = word
            idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        numericalized = [self.stoi["<bos>"]]
        numericalized.extend([self.stoi.get(token, self.stoi["<unk>"]) for token in tokenized_text])
        numericalized.append(self.stoi["<eos>"])
        return numericalized

# --- 2. Build Vocabulary from Training Data Only ---
TARGET_VOCAB_SIZE = 10000
vocab = Vocabulary()
vocab.build_vocabulary(train_df['captions'], vocab_size=TARGET_VOCAB_SIZE)
print(f"\nVocabulary built with {len(vocab)} total tokens.")
#lets print the whole vocab
print("\n--- Vocabulary ---")
for idx, word in vocab.itos.items():
    print(f"{idx}: {word}")


# --- 3. Analyze Caption Lengths to Justify max_length ---
print("\nAnalyzing caption lengths from clean data...")
all_lengths = []
for caption_list in tqdm(train_df['captions']):
    for sentence in caption_list:
        all_lengths.append(len(vocab.tokenizer(sentence)))

# Plotting the histogram
plt.figure(figsize=(12, 6))
sns.histplot(all_lengths, bins=30, kde=True)
plt.title('Histogram of Caption Lengths (in words) on Training Set')
plt.xlabel('Caption Length')
plt.ylabel('Frequency')
plt.grid(True)

p95 = int(np.percentile(all_lengths, 95))
p98 = int(np.percentile(all_lengths, 98))
plt.axvline(p95, color='red', linestyle='--', label=f'95th Percentile: {p95}')
plt.legend()
plt.show()


MAX_LENGTH = p98 + 2 # Adding 2 for <bos> and <eos>
print(f"Based on the corrected distribution, 98% of captions have {p98} words or fewer.")
print(f"We will set MAX_LENGTH = {MAX_LENGTH}.")

Counting word frequencies from clean data...



Vocabulary built with 2829 total tokens.

--- Vocabulary ---
0: <pad>
1: <bos>
2: <eos>
3: <unk>
4: a
5: are
6: green
7: many
8: trees
9: and
10: of
11: the
12: in
13: buildings
14: is
15: some
16: with
17: near
18: on
19: several
20: two
21: to
22: area
23: piece
24: around
25: river
26: by
27: an
28: road
29: it
30: located
31: surrounded
32: sides
33: close
34: large
35: cars
36: parking
37: playground
38: building
39: lot
40: white
41: there
42: residential
43: field
44: land
45: yellow
46: pond
47: 's
48: plants
49: square
50: parked
51: next
52: beach
53: ,
54: bridge
55: football
56: dense
57: airport
58: industrial
59: port
60: fields
61: viaduct
62: tanks
63: baseball
64: park
65: both
66: roads
67: desert
68: ocean
69: this
70: forest
71: curved
72: bare
73: houses
74: basketball
75: storage
76: planes
77: agricultural
78: meadows
79: boats
80: pieces
81: small
82: commercial
83: stadium
84: mountain
85: irregular
86: school
87: station
88: meadow
89: red
90: church
91: thre

<Figure size 1200x600 with 1 Axes>

Based on the corrected distribution, 98% of captions have 19 words or fewer.
We will set MAX_LENGTH = 21.


### Tokenizer bug: glued punctuation
Symptom: After building the initial vocabulary, an inspection revealed malformed tokens like boats.a, .near, and circle.some. This indicated that the tokenizer was not correctly separating words from punctuation when no space was present in the source text.
First version (flawed): The first version of the tokenizer was too simple. It tokenized using NLTK and then filtered for alphabetic tokens, failing to handle the combined word-punctuation tokens.

```python
# Flawed Version 1
@staticmethod
def tokenizer(text):
    text = text.lower()
    # This tokenizer doesn't split "boats.a" correctly
    return [tok for tok in nltk.tokenize.word_tokenize(text) if tok.isalpha()]
```
#### Unit Check (Proof of Bug): A simple test case demonstrates the failure.
```python
# --- Unit Check for Bug 1 ---
flawed_tokenizer_1 = Vocabulary.tokenizer # Using the flawed code above
test_sentence = "A few boats.a are here."
output_tokens = flawed_tokenizer_1(test_sentence)

print(f"Test Sentence: '{test_sentence}'")
print(f"Flawed Output: {output_tokens}")
print("Expected Output: ['a', 'few', 'boats', 'a', 'are', 'here']")
# The actual output would be ['a', 'few', 'are', 'here'], completely missing "boats" and "a".
```
Cause Analysis: The LLM's initial code made an implicit assumption that the input text would be well-formatted, with spaces separating all words and punctuation. It failed to account for common data entry errors found in real-world datasets.
The Fix (User-Guided Correction): The problem was resolved by introducing a regex-based cleaning step to force spaces around punctuation before tokenization.
Note: This fix itself was later found to be flawed, leading to Bug 2 .
### LLM Bug 2 : Overcorrection with Regex Breaks English Contractions
Symptom: After proposing a regex-based fix for Bug 1 , a new potential failure was identified through human review: the proposed fix would incorrectly handle standard English contractions like "don't" or "isn't".
Intermediate LLM-Generated Code (Flawed): The proposed fix for Bug 1 was to use re.sub to add spaces around punctuation. This was a classic example of an over-generalized rule.
```python
# Flawed Version 2
@staticmethod
def tokenizer(text):
    text = text.lower()
    # This regex incorrectly adds spaces in "don't" -> "don ' t"
    text = re.sub(r'([.,!?])', r' \1 ', text)
    # ... rest of the logic
```
#### Unit Check (Proof of Bug): A test case with a contraction proves the failure of this approach.
```python
def flawed_tokenizer_2(text):
    text = text.lower()
    # Incorrectly splits contractions
    text = re.sub(r"(['.,!?])", r' \1 ', text)
    return nltk.tokenize.word_tokenize(text)

test_word = "don't"
output_tokens = flawed_tokenizer_2(test_word)

print(f"Test Word: '{test_word}'")
print(f"Flawed Output: {output_tokens}")
print("Expected Output from NLTK: ['do', 'n't']")
# The actual output would be ['don', "'", 't'], which is linguistically incorrect.
```
### Cause Analysis: The LLM's second attempt was a brute-force approach that lacked linguistic nuance. It failed to recognize that an apostrophe (') is a special character that functions differently in contractions than a period (.) at the end of a sentence. This highlights the limitation of simple regex rules for complex natural language tasks.
The Final Fix (Corrected Code): The correct solution was to reverse the order of operations: let the expert (NLTK) handle the primary tokenization first, and then apply a more targeted regex cleanup pass on the resulting tokens.
```python
# Corrected Final Version
@staticmethod
def tokenizer(text):
    text = text.lower()
    initial_tokens = nltk.tokenize.word_tokenize(text)
    clean_tokens = []
    for token in initial_tokens:
        if token.isalpha() or token == "n't":
            clean_tokens.append(token)
            continue
        # This regex pass now only runs on "dirty" tokens, not the whole sentence.
        sub_tokens = re.findall(r'[a-z]+', token)
        if sub_tokens:
            clean_tokens.extend(sub_tokens)
    return clean_tokens
```

In [ ]:
class Vocabulary:
    def __init__(self):
        # The order of special tokens is important
        self.itos = {0: "<pad>", 1: "<bos>", 2: "<eos>", 3: "<unk>"}
        self.stoi = {"<pad>": 0, "<bos>": 1, "<eos>": 2, "<unk>": 3}

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer(text):
        text = text.lower()
        initial_tokens = nltk.tokenize.word_tokenize(text)
        clean_tokens = []
        for token in initial_tokens:
            if token.isalpha() or token == "n't":
                clean_tokens.append(token)
                continue
            sub_tokens = re.findall(r'[a-z]+', token)
            if sub_tokens:
                clean_tokens.extend(sub_tokens)
        return clean_tokens

    def build_vocabulary(self, captions_series, vocab_size):
        frequencies = Counter()
        idx = 4

        print("Counting word frequencies from clean data...")
        for caption_list in tqdm(captions_series):
            for sentence in caption_list:
                frequencies.update(self.tokenizer(sentence))

        print(f"Found {len(frequencies)} unique words.")
        most_common = frequencies.most_common(vocab_size - len(self.itos))

        for word, _ in most_common:
            self.stoi[word] = idx
            self.itos[idx] = word
            idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        numericalized = [self.stoi["<bos>"]]
        numericalized.extend([self.stoi.get(token, self.stoi["<unk>"]) for token in tokenized_text])
        numericalized.append(self.stoi["<eos>"])
        return numericalized

    def numericalize(self, text):
        tokenized_text = self.tokenizer(text)
        numericalized = [self.stoi["<bos>"]]
        numericalized.extend([self.stoi.get(token, self.stoi["<unk>"]) for token in tokenized_text])
        numericalized.append(self.stoi["<eos>"])
        return numericalized

# --- 2. Build Vocabulary from Training Data Only ---
TARGET_VOCAB_SIZE = 10000
vocab = Vocabulary()
vocab.build_vocabulary(train_df['captions'], vocab_size=TARGET_VOCAB_SIZE)
print(f"\nVocabulary built with {len(vocab)} total tokens.")
#lets print the whole vocab
print("\n--- Vocabulary ---")
for idx, word in vocab.itos.items():
    print(f"{idx}: {word}")


# --- 3. Analyze Caption Lengths to Justify max_length (with debugging) ---
print("\nAnalyzing caption lengths...")
all_lengths = []

# We will also look for and print outliers
print("\n--- Checking for Outlier Captions (length > 28) ---")
for caption_list in tqdm(train_df['captions']):
    for sentence in caption_list:
        length = len(vocab.tokenizer(sentence))
        all_lengths.append(len(vocab.tokenizer(sentence)))
        if length > 28: # Let's see any caption longer than 28 words
            print(f"OUTLIER FOUND (length={length}): {sentence}")
        all_lengths.append(length)
print("--- Outlier Check Complete ---")


# --- Plotting the histogram ---
plt.figure(figsize=(12, 6))
sns.histplot(all_lengths, bins=50, kde=True)
plt.title('Histogram of Caption Lengths (in words) on Training Set')
plt.xlabel('Caption Length')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

p90 = int(np.percentile(all_lengths, 90))
p95 = int(np.percentile(all_lengths, 95))
p98 = int(np.percentile(all_lengths, 98))

plt.axvline(p90, color='orange', linestyle='--', label=f'90th Percentile: {p90}')
plt.axvline(p95, color='red', linestyle='--', label=f'95th Percentile: {p95}')
plt.axvline(p98, color='green', linestyle='--', label=f'98th Percentile: {p98}')
plt.legend()
plt.show()


Counting word frequencies from clean data...


Found 2686 unique words.

Vocabulary built with 2690 total tokens.

--- Vocabulary ---
0: <pad>
1: <bos>
2: <eos>
3: <unk>
4: a
5: are
6: green
7: many
8: trees
9: and
10: of
11: the
12: in
13: buildings
14: is
15: some
16: with
17: near
18: on
19: several
20: two
21: to
22: area
23: piece
24: around
25: river
26: by
27: an
28: road
29: it
30: surrounded
31: located
32: sides
33: close
34: large
35: cars
36: parking
37: playground
38: building
39: lot
40: white
41: there
42: residential
43: field
44: land
45: yellow
46: s
47: pond
48: plants
49: square
50: parked
51: next
52: beach
53: bridge
54: football
55: dense
56: airport
57: industrial
58: port
59: fields
60: viaduct
61: tanks
62: baseball
63: park
64: both
65: roads
66: desert
67: ocean
68: this
69: forest
70: curved
71: bare
72: houses
73: basketball
74: storage
75: planes
76: agricultural
77: meadows
78: boats
79: pieces
80: small
81: commercial
82: stadium
83: mountain
84: irregular
85: school
86: station
87: meadow
88: red
8

OUTLIER FOUND (length=143): There are airport runways on the herbaceous field near which there is a cateno-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-catene-dethere are airport runways on the grassy field  next to which is a catenular terminal building lying on the tarmac  with some planes surrounding it


OUTLIER FOUND (length=30): next to a road group of factory buildings with the color of the roof of red and blue white is lying on the grassy ground with a unpaved path through
OUTLIER FOUND (length=31): on one side of the road that is along a row of trees there are buildings and cars while on another there is a lawn with a trail and trees along
OUTLIER FOUND (length=32): on one side of the road along which is a row of trees there are buildings and cars  while on the other there is a lawn with a path and trees along


OUTLIER FOUND (length=29): next to two houses there is a curved ploygon pond with a fountain at the corner of two roads and a curved path can be seen on the shore
OUTLIER FOUND (length=30): a school that has parking grounds on the sports grounds and a ploygon building is built along a road that is surrounded by trees and grass in a bare land


OUTLIER FOUND (length=34): four white columnar tanks are in the four pieces of square land next to a l sahped land combined by four pieces of black square land everyone of which has a columnar in it
OUTLIER FOUND (length=32): on this river we can see that the two bridges are connected to the banks of the river, and there are many inhabitants, houses and vehicles on both sides of the river
OUTLIER FOUND (length=29): the two bridges stood on top of the river, and both banks of the river had dense houses, and there was an island in the middle of the river
OUTLIER FOUND (length=31): this neighborhood is not only bustling and the environment is very good in addition to intensive high-rise buildings, there is a large area of the sports field and dense vegetation
OUTLIER FOUND (length=31): the playground is located in the quadrilateral surrounded by the buildings, and the buildings are too close to the playground, so that part of the playground is covered by the shadow
OUTLIER FOUND (length=3

<Figure size 1200x600 with 1 Axes>

<Figure size 640x480 with 1 Axes>

### Calculate how many captions we will be truncating with this choice

In [ ]:
CHOSEN_MAX_LENGTH = p98+2+2 # Adding 2 for bos/eos. adding 2 more to be safe.

num_captions = len(all_lengths)
num_truncated = sum(1 for length in all_lengths if length > CHOSEN_MAX_LENGTH - 2) # -2 for bos/eos
truncation_percent = (num_truncated / num_captions) * 100

print(f"\nThe raw 98th percentile is skewed by outliers to: {int(np.percentile(all_lengths, 98))}")
print(f"We will enforce a practical MAX_LENGTH of {CHOSEN_MAX_LENGTH}.")
print(f"This covers ~{100 - truncation_percent:.2f}% of all training captions.")
print(f"The remaining {truncation_percent:.2f}% will be truncated, which is an acceptable trade-off.")

# Set the final MAX_LENGTH variable for our dataset
MAX_LENGTH = CHOSEN_MAX_LENGTH


The raw 98th percentile is skewed by outliers to: 18
We will enforce a practical MAX_LENGTH of 22.
This covers ~99.22% of all training captions.
The remaining 0.78% will be truncated, which is an acceptable trade-off.


### Creating a train/val text stats table: vocab coverage, OOV %, length histogram

In [ ]:
def calculate_coverage(df, vocab_stoi):
    total_words = 0
    oov_words = 0
    for captions in tqdm(df['captions']):
        try:
            for sentence in captions:
                tokens = vocab.tokenizer(sentence)
                total_words += len(tokens)
                oov_words += sum(1 for word in tokens if word not in vocab_stoi)
        except (ValueError, SyntaxError):
            continue

    coverage = (1 - (oov_words / total_words)) * 100
    oov_percent = (oov_words / total_words) * 100
    return coverage, oov_percent

print("\nCalculating vocab stats for the TRAIN set...")
train_coverage, train_oov = calculate_coverage(train_df, vocab.stoi)

print("\nCalculating vocab stats for the VAL set...")
val_coverage, val_oov = calculate_coverage(val_df, vocab.stoi)


stats_data = {
    'Metric': ['Vocabulary Size', 'Chosen Max Length', 'Train Vocab Coverage', 'Train OOV %', 'Val Vocab Coverage', 'Val OOV %'],
    'Value': [f"{len(vocab)}", f"{MAX_LENGTH}", f"{train_coverage:.2f}%", f"{train_oov:.2f}%", f"{val_coverage:.2f}%", f"{val_oov:.2f}%"]
}
stats_df = pd.DataFrame(stats_data)

print("\n--- Train/Val Text Stats Table ---")
print(stats_df.to_string(index=False))


Calculating vocab stats for the TRAIN set...



Calculating vocab stats for the VAL set...



--- Train/Val Text Stats Table ---
              Metric   Value
     Vocabulary Size    2690
   Chosen Max Length      22
Train Vocab Coverage 100.00%
         Train OOV %   0.00%
  Val Vocab Coverage  99.07%
           Val OOV %   0.93%


# Baselines to Implement

## Building the CNN Encoder

In [ ]:
class CNNEncoder(nn.Module):
    """
    A CNN-based encoder to extract image features.
    """
    def __init__(self, model_name='resnet18', pretrained=True):
        """
        Args:
            model_name (str): The name of the CNN model to use ('resnet18' or 'mobilenet_v2').
            pretrained (bool): Whether to load pre-trained ImageNet weights.
        """
        super(CNNEncoder, self).__init__()
        self.model_name = model_name

        if model_name == 'resnet18':
            # Load pre-trained ResNet-18
            weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            self.cnn = models.resnet18(weights=weights)
            # The output feature dimension of ResNet-18's pooling layer is 512
            self.feature_dim = self.cnn.fc.in_features
            # Replace the final fully connected layer (the classifier) with an Identity layer
            self.cnn.fc = nn.Identity()
        elif model_name == 'mobilenet_v2':
            # Load pre-trained MobileNetV2
            weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
            self.cnn = models.mobilenet_v2(weights=weights)
            # The output feature dimension of MobileNetV2's pooling layer is 1280
            self.feature_dim = self.cnn.classifier[1].in_features
            # Replace the classifier block with an Identity layer
            self.cnn.classifier = nn.Identity()
        else:
            raise ValueError(f"Unsupported model: {model_name}. Choose 'resnet18' or 'mobilenet_v2'.")

    def forward(self, images):
        """
        Forward pass to extract features.
        Args:
            images (Tensor): A batch of images of shape (batch_size, 3, 224, 224).
        Returns:
            features (Tensor): A batch of feature vectors of shape (batch_size, feature_dim).
        """
        return self.cnn(images)

    def freeze_all_but_last_block(self):
        """
        Freezes all layers of the CNN except for the final convolutional block.
        This is used for end-to-end fine-tuning.
        """
        # First, freeze all parameters
        for param in self.cnn.parameters():
            param.requires_grad = False

        # Then, unfreeze the parameters of the last block
        if self.model_name == 'resnet18':
            # The last block in ResNet-18 is 'layer4'
            for param in self.cnn.layer4.parameters():
                param.requires_grad = True
        elif self.model_name == 'mobilenet_v2':
            # The last block in MobileNetV2 is 'features.18' (the inverted residual block)
            # Unfreezing the last few layers is a common strategy
            for param in self.cnn.features[-1].parameters():
                param.requires_grad = True

# --- Sanity Check ---
# Check if the model produces the correct feature shape
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA (NVIDIA GPU)")
# Check for MPS (Apple Silicon GPU)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
# Fallback to CPU
else:
    device = torch.device("cpu")
    print("Using CPU")

# Test ResNet-18
encoder_resnet = CNNEncoder(model_name='resnet18').to(device)
dummy_image_batch = torch.randn(4, 3, 224, 224).to(device) # Batch of 4 images
features = encoder_resnet(dummy_image_batch)
print(f"ResNet-18 output feature shape: {features.shape}") # Should be [4, 512]

# Test MobileNetV2
encoder_mobilenet = CNNEncoder(model_name='mobilenet_v2').to(device)
features = encoder_mobilenet(dummy_image_batch)
print(f"MobileNetV2 output feature shape: {features.shape}") # Should be [4, 1280]

Using MPS (Apple Silicon GPU)
ResNet-18 output feature shape: torch.Size([4, 512])
MobileNetV2 output feature shape: torch.Size([4, 1280])


## Feature Caching

In [ ]:
class ImageOnlyDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_bytes = ast.literal_eval(row['image'])['bytes']
        # The filename in the 'filename' column might be like 'rsicd_images/airport_1.jpg'
        # We just want the 'airport_1.jpg' part.
        image_id = os.path.basename(row['filename'])

        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, image_id

def save_features(encoder, df, split_name, batch_size=32, device='cuda'):
    """
    Function to pre-compute and save features for a given data split.
    """
    save_dir = f'./{split_name}_resnet_features'
    os.makedirs(save_dir, exist_ok=True)
    print(f"Saving features for '{split_name}' split to '{save_dir}'...")

    dataset = ImageOnlyDataset(df, transform=image_transforms)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    encoder.eval()

    # Disable gradient calculation
    with torch.no_grad():
        for images, image_ids in tqdm(dataloader, desc=f"Processing {split_name}"):
            images = images.to(device)
            # Get features from the encoder
            features = encoder(images) # Shape: [batch_size, feature_dim]

            # Save each feature vector individually
            for feature, img_id in zip(features, image_ids):
                # The image_id is 'airport_1.jpg'. We'll save as 'airport_1.pt'.
                base_filename = os.path.splitext(img_id)[0]
                save_path = os.path.join(save_dir, f"{base_filename}.pt")
                # Move feature to CPU before saving to avoid GPU memory issues in the main process
                torch.save(feature.cpu(), save_path)
    print("Done.")


encoder_resnet = CNNEncoder(model_name='resnet18').to(device)
encoder_mobilenet = CNNEncoder(model_name='mobilenet_v2').to(device)

save_features(encoder_resnet, train_df, 'train', device=device)
save_features(encoder_resnet, val_df, 'val', device=device)
save_features(encoder_resnet, test_df, 'test', device=device)
# save_features(encoder_mobilenet, train_df, 'train', device=device)
# save_features(encoder_mobilenet, val_df, 'val', device=device)
# save_features(encoder_mobilenet, test_df, 'test', device=device)

Saving features for 'train' split to './train_resnet_features'...


Done.
Saving features for 'val' split to './val_resnet_features'...


Done.
Saving features for 'test' split to './test_resnet_features'...


Done.


## Freeze all but last block

In [18]:
# 1. Instantiate the encoder
encoder_resnet = CNNEncoder(model_name='resnet18').to(device)

# 2. Call the method to freeze the early layers and unfreeze the last block
encoder_resnet.freeze_all_but_last_block()

# 3. Verify that it worked (optional but good practice)
for name, param in encoder_resnet.named_parameters():
    if 'layer4' in name: # Check the last block of ResNet-18
        assert param.requires_grad == True
    elif 'fc' not in name: # fc is identity, has no params
        assert param.requires_grad == False
print("\n✅ Success: Resnet encoder correctly set up for fine-tuning.")



# 1. Instantiate the encoder
encoder_mobilenet = CNNEncoder(model_name='mobilenet_v2').to(device)

# 2. Call the method to freeze the early layers and unfreeze the last block
encoder_mobilenet.freeze_all_but_last_block()

unfrozen_params_found = False
for name, param in encoder_mobilenet.named_parameters():
    if 'features.18' in name:
        # Assert that parameters in the last block are UNFROZEN
        assert param.requires_grad == True, f"Parameter {name} should be unfrozen!"
        unfrozen_params_found = True
    else:
        # Assert that all other parameters are FROZEN
        assert param.requires_grad == False, f"Parameter {name} should be frozen!"

if unfrozen_params_found:
    print("\n✅ Success: MobileNetV2 encoder correctly set up for fine-tuning.")
else:
    print("\n❌ Failure: No parameters were unfrozen. Check the layer name in the freeze method.")


✅ Success: Resnet encoder correctly set up for fine-tuning.

✅ Success: MobileNetV2 encoder correctly set up for fine-tuning.


## Building the LSTM Decoder

In [ ]:
class LSTMDecoder(nn.Module):
    def __init__(self, encoder_dim, embed_dim, hidden_dim, vocab_size, num_layers, dropout_p=0.5):
        super(LSTMDecoder, self).__init__()
        # Store these for use in generate_caption
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        
        # The robust architecture: LSTM input is a concatenation of word embedding and image feature
        self.lstm = nn.LSTM(embed_dim + encoder_dim, hidden_dim, num_layers, batch_first=True)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, features, captions):
        captions = captions[:, :-1]
        embeddings = self.embedding(captions)
        features_repeated = features.unsqueeze(1).repeat(1, embeddings.size(1), 1)
        inputs = torch.cat((embeddings, features_repeated), dim=2)
        inputs = self.dropout(inputs)
        lstm_out, _ = self.lstm(inputs)
        outputs = self.fc(lstm_out)
        return outputs

    def generate_caption(self, features, vocab, max_len=24):
        self.eval()
        caption_indices = [vocab.stoi['<bos>']]
        
        with torch.no_grad():
            # Manually create the initial hidden and cell states with the correct shape.
            # Shape: (num_layers, batch_size, hidden_dim). Here batch_size is 1.
            hidden = torch.zeros(self.num_layers, 1, self.hidden_dim).to(features.device)
            cell = torch.zeros(self.num_layers, 1, self.hidden_dim).to(features.device)
            states = (hidden, cell)
            # ---------------------------------
            
            for _ in range(max_len):
                # Get the last predicted word index
                last_word_idx = torch.tensor([caption_indices[-1]], device=features.device)
                
                # Get its embedding
                embedding = self.embedding(last_word_idx) # Shape: (1, embed_dim)

                # Create the input by concatenating the new embedding and the image feature
                # features has shape (1, encoder_dim), embedding has shape (1, embed_dim)
                inputs = torch.cat((embedding, features), dim=1).unsqueeze(1) # Final shape: (1, 1, embed_dim + encoder_dim)
                
                # We now always pass a valid tensor tuple as the state
                lstm_out, states = self.lstm(inputs, states)
                
                outputs = self.fc(lstm_out.squeeze(1))
                predicted_idx = outputs.argmax(1)
                
                caption_indices.append(predicted_idx.item())
                
                if predicted_idx.item() == vocab.stoi['<eos>']:
                    break
                    
        self.train()
        # Return the final caption, skipping the <bos> token
        return [vocab.itos[idx] for idx in caption_indices[1:] if idx != vocab.stoi['<eos>']]

    def generate_caption_beam_search(self, features, vocab, max_len=24, beam_width=3):
        self.eval()
        with torch.no_grad():
            # Manually create initial states
            hidden = torch.zeros(self.num_layers, 1, self.hidden_dim).to(features.device)
            cell = torch.zeros(self.num_layers, 1, self.hidden_dim).to(features.device)
            initial_states = (hidden, cell)
            
            start_token_idx = vocab.stoi['<bos>']
            start_seq = torch.tensor([start_token_idx]).to(features.device)
            beam = [(start_seq, 0.0, initial_states)]
            completed_sequences = []

            for _ in range(max_len):
                new_beam = []
                for seq, log_prob, states in beam:
                    last_token = seq[-1]
                    if last_token.item() == vocab.stoi['<eos>']:
                        completed_sequences.append((seq, log_prob))
                        continue

                    last_word_embedding = self.embedding(last_token).unsqueeze(0) # Shape: (1, embed_dim)
                    inputs = torch.cat((last_word_embedding, features), dim=1).unsqueeze(1) # Final shape: (1, 1, embed_dim + encoder_dim)
                    
                    lstm_out, new_states = self.lstm(inputs, states)
                    outputs = self.fc(lstm_out.squeeze(1))
                    log_probs = nn.functional.log_softmax(outputs, dim=1)
                    
                    top_log_probs, top_indices = log_probs.topk(beam_width, dim=1)
                    
                    for i in range(beam_width):
                        next_word_idx = top_indices[0, i]
                        next_log_prob = top_log_probs[0, i].item()
                        new_seq = torch.cat([seq, next_word_idx.unsqueeze(0)])
                        new_beam.append((new_seq, log_prob + next_log_prob, new_states))
                
                if not new_beam: break
                beam = sorted(new_beam, key=lambda x: x[1], reverse=True)[:beam_width]

            completed_sequences.extend([(s, lp) for s, lp, _ in beam])
            if not completed_sequences: # Handle case where no sequences complete
                if not beam: return [] # Handle empty beam case
                best_seq, best_log_prob, _ = beam[0]
            else:
                best_seq, best_log_prob = sorted(completed_sequences, key=lambda x: x[1] / len(x[0]), reverse=True)[0]
            
            caption = [vocab.itos[idx.item()] for idx in best_seq[1:] if idx.item() != vocab.stoi['<eos>']]

        self.train()
        return caption

In [ ]:
class CaptionDataset(Dataset):
    def __init__(self, df, vocab, feature_dir, max_len):
        self.df = df
        self.vocab = vocab
        self.feature_dir = feature_dir
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Load the pre-computed image feature
        base_filename = os.path.splitext(os.path.basename(row['filename']))[0]
        feature_path = os.path.join(self.feature_dir, f"{base_filename}.pt")
        feature = torch.load(feature_path)
        
        all_captions = row['captions']

        # Handle rare cases where the caption list might be empty after cleaning
        if not all_captions:
            all_captions = ["a remote sensing image"] # Providing a generic fallback

        # Randomly choose one caption for this training step
        caption = random.choice(all_captions)

        # Numericalize and pad the chosen caption
        numericalized_caption = self.vocab.numericalize(caption)
        padded_caption = torch.full((self.max_len,), self.vocab.stoi['<pad>'], dtype=torch.long)

        end = min(len(numericalized_caption), self.max_len)
        padded_caption[:end] = torch.tensor(numericalized_caption[:end])

        return feature, padded_caption, all_captions


def collate_fn(batch):
    features, padded_captions, all_captions_strs = zip(*batch)
    features = torch.stack(features, 0)
    padded_captions = torch.stack(padded_captions, 0)
    return features, padded_captions, all_captions_strs


# --- 3. Hyperparameters and Setup ---

# Model Hyperparameters
ENCODER_DIM = 512  # This must match the output of our ResNet-18 encoder
EMBED_DIM = 512
HIDDEN_DIM = 512
VOCAB_SIZE = len(vocab)
NUM_LAYERS = 2 

# Training Hyperparameters
LEARNING_RATE = 2e-4
NUM_EPOCHS = 20
BATCH_SIZE = 64

# Setup
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA (NVIDIA GPU)")
# Check for MPS (Apple Silicon GPU)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
# Fallback to CPU
else:
    device = torch.device("cpu")
    print("Using CPU")
print(f"Using device: {device}")

# Directories for cached features
train_feature_dir = './train_resnet_features'
val_feature_dir = './val_resnet_features'

# Instantiate Datasets
train_dataset = CaptionDataset(train_df, vocab, train_feature_dir, MAX_LENGTH)
val_dataset = CaptionDataset(val_df, vocab, val_feature_dir, MAX_LENGTH)

# Instantiate DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# Instantiate the Model, Loss, and Optimizer
model = EncoderDecoder(ENCODER_DIM, EMBED_DIM, HIDDEN_DIM, VOCAB_SIZE, NUM_LAYERS).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi['<pad>'])
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# --- 4. Training and Evaluation Loops ---

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for features, captions, _ in tqdm(loader, desc="Training"):
        features = features.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()
        outputs = model(features, captions)
        loss = criterion(outputs.view(-1, VOCAB_SIZE), captions[:, 1:].reshape(-1))
        loss.backward()

        # Clip the gradients to prevent them from exploding and overpowering the image features
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # ---------------------

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, vocab):
    model.eval()
    print("\n--- Generating Example Captions ---")

    with torch.no_grad():
        # Get one batch from the validation set
        try:
            features, _, all_captions_strs = next(iter(loader))
        except StopIteration:
            print("Validation loader is empty.")
            return

        features = features.to(device)

        
        for i in range(min(5, len(features))):
            feature = features[i].unsqueeze(0)
            generated_caption_words = model.decoder.generate_caption(feature, vocab, max_len=MAX_LENGTH)
            generated_caption = ' '.join(generated_caption_words)

            print(f"\nExample {i+1}:")
            print(f"  Generated: {generated_caption}")

            gt_captions = all_captions_strs[i]

            print(f"  Ground Truth 1: {gt_captions[0]}")

            if len(gt_captions) > 1:
                print(f"  Ground Truth 2: {gt_captions[1]}")

    print("--- Evaluation Complete ---\n")

def evaluate_with_beam_search(model, loader, vocab):
    model.eval()
    print("\n--- Generating Example Captions (with Beam Search, k=3) ---")

    with torch.no_grad():
        features, _, all_captions_strs = next(iter(loader))
        features = features.to(device)

        for i in range(min(5, len(features))):
            feature = features[i].unsqueeze(0)

            generated_caption_words = model.decoder.generate_caption_beam_search(feature, vocab, max_len=MAX_LENGTH, beam_width=3)

            generated_caption = ' '.join(generated_caption_words)

            print(f"\nExample {i+1}:")
            print(f"  Generated: {generated_caption}")

            gt_captions = all_captions_strs[i]
            print(f"  Ground Truth 1: {gt_captions[0]}")
            if len(gt_captions) > 1:
                print(f"  Ground Truth 2: {gt_captions[1]}")
    print("--- Evaluation Complete ---\n")


for epoch in range(1, NUM_EPOCHS + 1):
    avg_train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch}/{NUM_EPOCHS}], Average Training Loss: {avg_train_loss:.4f}, Current LR: {current_lr}")

    evaluate(model, val_loader, vocab)
    evaluate_with_beam_search(model, val_loader, vocab)


print("Training finished.")





Using MPS (Apple Silicon GPU)
Using device: mps


Epoch [1/20], Average Training Loss: 4.8496, Current LR: 0.0002

--- Generating Example Captions ---

Example 1:
  Generated: many green are are a a
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many green are are a a
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green are are a a
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is lying on the bareland  near which there are some square buildings

Example 4:
  Gene

Epoch [2/20], Average Training Loss: 3.7496, Current LR: 0.0002

--- Generating Example Captions ---

Example 1:
  Generated: many green trees are in a piece of a
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many green trees are in a piece of a
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are in a piece of a
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is lying on the bareland  near which there ar

Epoch [3/20], Average Training Loss: 3.0286, Current LR: 0.0002

--- Generating Example Captions ---

Example 1:
  Generated: many green trees are in a piece of green trees
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many green trees are in a piece of green trees
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are in a piece of green trees
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is lying on the

Epoch [4/20], Average Training Loss: 2.6908, Current LR: 0.0002

--- Generating Example Captions ---

Example 1:
  Generated: many buildings and some green trees are around a playground
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many buildings and some green trees are around a playground
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are around a playground
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a ru

...

Epoch [16/20], Average Training Loss: 2.0705, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are around a playground with a playground
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many buildings are around a park with many green trees and a pond
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are around a playground
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on a

Epoch [17/20], Average Training Loss: 2.0873, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are around a playground with a playground
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many buildings are around a park with many green trees
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are around a playground
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connecte

Epoch [18/20], Average Training Loss: 2.0853, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are around a playground with a playground
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many buildings are around a park with many green trees and a pond
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are around a playground
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on a

Epoch [19/20], Average Training Loss: 2.0686, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are around a playground with a playground
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many buildings are around a park with many green trees
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are around a building
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected 

Epoch [20/20], Average Training Loss: 2.0710, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are around a playground with a playground
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many buildings are around a park with many green trees
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are around a playground
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connecte

In [46]:
torch.save(model.state_dict(), '/Users/arnavagarwal/Documents/Sem7/EE782/Assignment1/cnn_lstm.pth')
print("Model weights saved to cnn_lstm.pth")

Model weights saved to cnn_lstm.pth


## Building Transformer Decoder

In [ ]:
import torch
import torch.nn as nn
import math
from tqdm import tqdm

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=100):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        # Change from (max_len, 1, d_model) to (1, max_len, d_model) to be batch_first compatible
        self.register_buffer('pe', pe.transpose(0, 1))

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class TransformerDecoderModel(nn.Module):
    def __init__(self, encoder_dim, d_model, nhead, num_layers, vocab_size, memory_len=4, dropout_p=0.5):
        super(TransformerDecoderModel, self).__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.memory_projection = nn.Linear(encoder_dim, memory_len * d_model)
        self.memory_layernorm = nn.LayerNorm(d_model)
        self.memory_len = memory_len
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, features, captions, tgt_mask, tgt_padding_mask):
        tgt = captions # Already sliced to exclude <eos>
        memory = self.memory_projection(features).view(-1, self.memory_len, self.d_model)
        memory = self.memory_layernorm(memory)
        tgt_embed = self.embedding(tgt) * math.sqrt(self.d_model)
        tgt_embed = self.pos_encoder(tgt_embed)
        output = self.transformer_decoder(
            tgt=tgt_embed,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )
        return self.fc(output)

    def generate_caption(self, features, vocab, max_len=24):
        self.eval()
        with torch.no_grad():
            memory = self.memory_projection(features).view(-1, self.memory_len, self.d_model)
            memory = self.memory_layernorm(memory)
            generated_seq = torch.tensor([vocab.stoi['<bos>']]).long().to(features.device).unsqueeze(0)
            for _ in range(max_len - 1):
                tgt_len = generated_seq.size(1)
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len).to(features.device)
                tgt_embed = self.embedding(generated_seq) * math.sqrt(self.d_model)
                tgt_embed = self.pos_encoder(tgt_embed)
                output = self.transformer_decoder(tgt_embed, memory, tgt_mask=tgt_mask)
                last_word_logits = self.fc(output[:, -1, :])
                predicted_idx = last_word_logits.argmax(1)
                generated_seq = torch.cat([generated_seq, predicted_idx.unsqueeze(0)], dim=1)
                if predicted_idx.item() == vocab.stoi['<eos>']:
                    break
        self.train()
        caption_indices = generated_seq.squeeze(0).cpu().tolist()
        return [vocab.itos[idx] for idx in caption_indices[1:] if idx != vocab.stoi['<eos>']]

class EncoderDecoderTransformer(nn.Module):
    def __init__(self, encoder_dim, d_model, nhead, num_layers, vocab_size):
        super(EncoderDecoderTransformer, self).__init__()
        self.decoder = TransformerDecoderModel(encoder_dim, d_model, nhead, num_layers, vocab_size)
    def forward(self, features, captions, tgt_mask, tgt_padding_mask):
        return self.decoder(features, captions, tgt_mask, tgt_padding_mask)

# --- 2. Hyperparameters and Setup ---

# Model Hyperparameters
ENCODER_DIM = 512  # From ResNet-18
D_MODEL = 512      # The main dimension of the Transformer
NHEAD = 8          # Number of attention heads
NUM_LAYERS_TR = 4  # Number of decoder layers
VOCAB_SIZE = len(vocab)

# Training Hyperparameters
LEARNING_RATE_TR = 2e-4
NUM_EPOCHS_TR = 20 
BATCH_SIZE = 64

# Setup
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA (NVIDIA GPU)")
# Check for MPS (Apple Silicon GPU)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
# Fallback to CPU
else:
    device = torch.device("cpu")
    print("Using CPU")
print(f"Using device for Transformer: {device}")

# Instantiate the Model, Loss, and Optimizer
model_tr = EncoderDecoderTransformer(ENCODER_DIM, D_MODEL, NHEAD, NUM_LAYERS_TR, VOCAB_SIZE).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi['<pad>'])
optimizer_tr = torch.optim.Adam(model_tr.parameters(), lr=LEARNING_RATE_TR)
scheduler_tr = torch.optim.lr_scheduler.StepLR(optimizer_tr, step_size=7, gamma=0.1)
# --- 3. Training and Evaluation Loops for Transformer ---

def create_causal_mask(size, device):
    return nn.Transformer.generate_square_subsequent_mask(size, device=device)

def create_padding_mask(seq, pad_idx, device):
    return (seq == pad_idx).to(device)

def train_one_epoch_transformer(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for features, captions, _ in tqdm(loader, desc="Training Transformer"):
        features, captions = features.to(device), captions.to(device)
        optimizer.zero_grad()
        
        tgt_in = captions[:, :-1]
        tgt_out = captions[:, 1:]
        
        tgt_mask = create_causal_mask(tgt_in.size(1), device)
        tgt_padding_mask = create_padding_mask(tgt_in, vocab.stoi['<pad>'], device)

        outputs = model(features, tgt_in, tgt_mask, tgt_padding_mask)
        loss = criterion(outputs.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(loader)

def evaluate_transformer(model, loader, vocab):
    model.eval()
    print("\n--- Generating Example Captions (Transformer) ---")
    with torch.no_grad():
        features, _, all_captions_strs = next(iter(loader))
        features = features.to(device)
        for i in range(min(5, len(features))):
            feature = features[i].unsqueeze(0)
            generated_caption_words = model.decoder.generate_caption(feature, vocab, max_len=MAX_LENGTH)
            generated_caption = ' '.join(generated_caption_words)
            print(f"\nExample {i+1}:")
            print(f"  Generated: {generated_caption}")
            gt_captions = all_captions_strs[i]
            print(f"  Ground Truth 1: {gt_captions[0]}")
            if len(gt_captions) > 1: print(f"  Ground Truth 2: {gt_captions[1]}")
    print("--- Evaluation Complete ---\n")



print("\n--- Starting Transformer Model Training ---")
for epoch in range(1, NUM_EPOCHS_TR + 1):
    avg_train_loss = train_one_epoch_transformer(model_tr, train_loader, optimizer_tr, criterion)
    scheduler_tr.step()
    current_lr = optimizer_tr.param_groups[0]['lr']
    
    print(f"Epoch [{epoch}/{NUM_EPOCHS_TR}], Loss: {avg_train_loss:.4f}, LR: {current_lr}")
    evaluate_transformer(model_tr, val_loader, vocab)

print("Transformer training finished.")
torch.save(model_tr.state_dict(), 'cnn_transformer_baseline.pth')
print("Transformer model weights saved to cnn_transformer_baseline.pth")

Using MPS (Apple Silicon GPU)
Using device for Transformer: mps

--- Starting Transformer Model Training ---


  warnings.warn(


Epoch [1/20], Loss: 3.0518, LR: 0.0002

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: many green trees and a playground are close to a road
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many planes are parked in an airport near a road
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees and a road are close to a piece of green meadow
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway

Epoch [2/20], Loss: 2.0131, LR: 0.0002

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: many buildings and some green trees are in two sides of a railway station
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many planes are parked in an airport near some buildings and some green trees
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: a building is near a piece of yellow beach
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked o

Epoch [3/20], Loss: 1.7940, LR: 0.0002

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some buildings and green trees are in two sides of a railway station
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many planes are in an airport near some green trees and several buildings
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: some planes are in an airport near a parking lot
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on a

...

Epoch [16/20], Loss: 1.1887, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near some green trees
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many planes are parked in an airport near some buildings
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: a road is near a piece of bare land
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is ly

Epoch [17/20], Loss: 1.1822, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near some green trees
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many planes are parked in an airport near some buildings
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: a road is near a piece of bare land
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is ly

Epoch [18/20], Loss: 1.1909, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near some green trees
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many planes are parked in an airport near some buildings
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: a road is near a piece of bare land
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is ly

Epoch [19/20], Loss: 1.1895, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near some green trees
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many planes are parked in an airport near some buildings
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: a road is near a piece of bare land
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is ly

Epoch [20/20], Loss: 1.1967, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near some green trees
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many planes are parked in an airport near some buildings
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: a road is near a piece of bare land
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is ly

# Debugging notes

Four issues hit while building the pipeline, each with the check that exposed it and the fix. 

---

## Bug 1 : ValueError from Pipeline Refactoring Causes Silent Vocabulary Failure
Symptom
After changing the captions column from strings → lists via a cleaning function, the vocabulary builder failed silently.
It reported 0 unique words and crashed with IndexError when calculating percentiles.
### Unit Check (Proof of Bug)
```python
train_df['captions'] = train_df['captions'].apply(clean_and_split_captions)

print(f"Data type after cleaning: {type(train_df['captions'].iloc[0])}")

try:
    ast.literal_eval(train_df['captions'].iloc[0])
except ValueError as e:
    print(f"\nBug confirmed. ast.literal_eval fails with error: {e}")
```
Result: captions was already a list, and ast.literal_eval raised ValueError.
## Analysis
A pipeline refactoring bug: the old parsing logic (ast.literal_eval) was kept even though the upstream cleaning step already returned lists.
✅ Fix
Remove unnecessary parsing and let the loop iterate directly:
```python
# Corrected loop inside build_vocabulary
def build_vocabulary(self, captions_series, vocab_size):
    for caption_list in tqdm(captions_series):  # already lists
        for sentence in caption_list:
            frequencies.update(self.tokenizer(sentence))
```

---

## Bug 2 : IndexError in Evaluation Loop Due to Brittle Data Assumption
Symptom 

Training ran fine, but evaluate() crashed with IndexError: list index out of range when printing the second ground truth caption (gt_captions[1]).
### Unit Check (Proof of Bug)
```python
buggy_row_found = False
for i, row in val_df.iterrows():
    if len(row['captions']) < 2:
        print(f"Bug confirmed: Row {i} has only {len(row['captions'])} caption(s).")
        buggy_row_found = True
        break
if not buggy_row_found: print("No buggy rows found.")
```
Result: Some rows had only 1 caption, confirming the assumption was invalid.
🔎 Analysis
The LLM-generated code assumed every image had ≥2 captions, which is brittle and unrealistic in noisy datasets.
✅ Fix
Make the evaluation logic defensive:
# Corrected logic inside evaluate
```python
gt_captions = all_captions_strs[i]
print(f"  Ground Truth 1: {gt_captions[0]}")

if len(gt_captions) > 1:
    print(f"  Ground Truth 2: {gt_captions[1]}")
```
---

## Bug 3 : Naive Regex for Tokenization Breaks English Contractions
Symptom

A regex fix for punctuation caused incorrect splitting of English contractions:
e.g., "don't" → ['don', "'", 't']
### Unit Check (Proof of Bug)
```python
import re

def flawed_tokenizer_intermediate(text):
    text = text.lower()
    text = re.sub(r"(['.,!?])", r' \1 ', text)  # overly aggressive regex
    return nltk.tokenize.word_tokenize(text)

test_word = "don't"
print(flawed_tokenizer_intermediate(test_word))
Result: ['don', "'", 't'] (wrong)
Correct: ['do', "n't"]
```
🔎 Analysis
Regex treated all apostrophes like punctuation, destroying contractions. The LLM fix solved one bug (stuck punctuation) but introduced a worse one.
✅ Fix
Let nltk.word_tokenize handle contractions, then apply targeted cleanup:
### Corrected final tokenizer
```python
def final_tokenizer(text):
    text = text.lower()
    initial_tokens = nltk.tokenize.word_tokenize(text)
    # Apply targeted cleanup on dirty tokens...
    return initial_tokens
```

---

## Bug 4
### Behavior Analysis: Transformer Model Generates Short Captions



## 👀 Acknowledging the Behavior
It was observed that while the **Transformer model** achieved a very low training loss and generated **semantically correct captions**, the outputs were **consistently shorter** than the human-written ground truth references.



## 💡 Hypothesizing the Cause
This behavior is a **well-known artifact** of sequence-to-sequence models trained with **standard cross-entropy loss**:

- The model is incentivized to predict the `<eos>` (end-of-sequence) token **as early as possible** to minimize cumulative loss.  
- This shortsightedness is further **amplified by greedy decoding**, which favors immediate high-probability tokens instead of exploring longer continuations.  
- As a result, the model settles for short but locally optimal captions, at the cost of **length and richness**.



## 🛠️ Proposing a Solution
A potential fix (not implemented in this baseline) would be to apply a **length penalty during inference**:

- This adjustment artificially **increases the score of longer candidate sequences** during beam search.  
- Encourages the model to produce **more detailed captions**.  
- Aligns generated outputs more closely with the **length, style, and richness** of human references.

---


## Bug 5 LLM Debugging Diary: Overcoming LSTM Mode Collapse and Architectural Flaws

### Initial Symptom: The Model Ignores the Image (Mode Collapse)
After several epochs of training the initial LSTM model, the evaluation loop showed a critical failure: the model generated the exact same generic caption for every single validation image, regardless of its content.

### Evidence (Proof of Failure):
The evaluation output was the definitive proof. The generated captions for 5 different validation images were identical, proving the model was not using the unique image features provided as input.

### Output from the initial, flawed model:
Example 1 Generated: many green trees are around a building

Example 2 Generated: many green trees are around a building

Example 3 Generated: many green trees are around a building

This occurred even while the training loss was consistently decreasing, which masked the severity of the problem.

### Initial Hypothesis:
The language modeling gradients from the LSTM were overpowering the weak, initial signal from the image features, causing the optimizer to find a "safe" local minimum by producing a high-frequency, generic sentence.

### Iteration 1: Attempting to Fix with Hyperparameter Tuning
Two standard techniques were attempted first to solve what was believed to be a training instability.

1. Applying Gradient Clipping:

Action: Added torch.nn.utils.clip_grad_norm_ to the training loop to prevent exploding gradients.
Result: Failure. The mode collapse persisted, indicating the problem was more deeply rooted than just gradient spikes.

2. Reducing Learning Rate & Adding a Scheduler:

Action: The learning rate was significantly reduced (from 2e-4 to 3e-5) and a StepLR scheduler was added to encourage more stable, smaller optimization steps.
Result: Failure. The problem still persisted. This critical result proved the issue was not with the training dynamics but with the model's architecture.

### Deeper Diagnosis: The "Vanishing Image" Problem
A deeper analysis revealed an architectural flaw. The model design only injected the image feature once, to set the LSTM's initial hidden state. This "memory" of the image was too weak. The powerful, step-by-step signal from teacher forcing quickly overwhelmed this initial state, causing the model to effectively "forget" the image after the first few words.

### The Architectural Fix: Constant Image Conditioning
The LSTMDecoder was fundamentally re-architected to force it to consider the image at every single time step. This was achieved by concatenating the image feature vector with the word embedding at each step of the sequence.

```python
# The key change in the NEW LSTMDecoder.forward method:
# The LSTM's input is now a concatenation of both sources of information
inputs = torch.cat((embeddings, features_repeated), dim=2)
lstm_out, _ = self.lstm(inputs)
This architectural change ensures the image signal is persistent and cannot be ignored.
```


# Experiments & Extensions

## Backbone swap: ResNet18 ↔ MobileNet;

In [ ]:
# --- 3. Hyperparameters and Setup ---

# Model Hyperparameters
ENCODER_DIM = 1280 
EMBED_DIM = 512
HIDDEN_DIM = 512
VOCAB_SIZE = len(vocab)
NUM_LAYERS = 2 

# Training Hyperparameters
LEARNING_RATE = 2e-4
NUM_EPOCHS = 20
BATCH_SIZE = 64

# Setup
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA (NVIDIA GPU)")
# Check for MPS (Apple Silicon GPU)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
# Fallback to CPU
else:
    device = torch.device("cpu")
    print("Using CPU")
print(f"Using device: {device}")

# Directories for cached features
train_feature_dir = './train_mobilenet_features'
val_feature_dir = './val_mobilenet_features'

# Instantiate Datasets
train_dataset = CaptionDataset(train_df, vocab, train_feature_dir, MAX_LENGTH)
val_dataset = CaptionDataset(val_df, vocab, val_feature_dir, MAX_LENGTH)

# Instantiate DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# Instantiate the Model, Loss, and Optimizer
model = EncoderDecoder(ENCODER_DIM, EMBED_DIM, HIDDEN_DIM, VOCAB_SIZE, NUM_LAYERS).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi['<pad>'])
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# --- 4. Training and Evaluation Loops ---

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for features, captions, _ in tqdm(loader, desc="Training"):
        features = features.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()
        outputs = model(features, captions)
        loss = criterion(outputs.view(-1, VOCAB_SIZE), captions[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, vocab):
    model.eval()
    print("\n--- Generating Example Captions ---")

    with torch.no_grad():
        # Get one batch from the validation set
        try:
            features, _, all_captions_strs = next(iter(loader))
        except StopIteration:
            print("Validation loader is empty.")
            return

        features = features.to(device)

        for i in range(min(5, len(features))): 
            feature = features[i].unsqueeze(0)
            generated_caption_words = model.decoder.generate_caption(feature, vocab, max_len=MAX_LENGTH)
            generated_caption = ' '.join(generated_caption_words)

            print(f"\nExample {i+1}:")
            print(f"  Generated: {generated_caption}")

            gt_captions = all_captions_strs[i]

            print(f"  Ground Truth 1: {gt_captions[0]}")

            if len(gt_captions) > 1:
                print(f"  Ground Truth 2: {gt_captions[1]}")

    print("--- Evaluation Complete ---\n")

def evaluate_with_beam_search(model, loader, vocab):
    model.eval()
    print("\n--- Generating Example Captions (with Beam Search, k=3) ---")

    with torch.no_grad():
        features, _, all_captions_strs = next(iter(loader))
        features = features.to(device)

        for i in range(min(5, len(features))):
            feature = features[i].unsqueeze(0)

            generated_caption_words = model.decoder.generate_caption_beam_search(feature, vocab, max_len=MAX_LENGTH, beam_width=3)
        

            generated_caption = ' '.join(generated_caption_words)

            print(f"\nExample {i+1}:")
            print(f"  Generated: {generated_caption}")

            gt_captions = all_captions_strs[i]
            print(f"  Ground Truth 1: {gt_captions[0]}")
            if len(gt_captions) > 1:
                print(f"  Ground Truth 2: {gt_captions[1]}")
    print("--- Evaluation Complete ---\n")



for epoch in range(1, NUM_EPOCHS + 1):
    avg_train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch}/{NUM_EPOCHS}], Average Training Loss: {avg_train_loss:.4f}, Current LR: {current_lr}")

    # Run evaluation to see qualitative results
    evaluate(model, val_loader, vocab)
    evaluate_with_beam_search(model, val_loader, vocab)


print("Training finished.")

Using MPS (Apple Silicon GPU)
Using device: mps


Epoch [1/20], Average Training Loss: 4.8417, Current LR: 0.0002

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are are a
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many buildings are are a
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many buildings are are a
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is lying on the bareland  near which there are some square buildings

Example 4:


Epoch [2/20], Average Training Loss: 3.8545, Current LR: 0.0002

--- Generating Example Captions ---

Example 1:
  Generated: many green trees are in a piece of
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many buildings are in a piece of green trees
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are in a piece of
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is lying on the bareland  near which ther

Epoch [3/20], Average Training Loss: 3.1218, Current LR: 0.0002

--- Generating Example Captions ---

Example 1:
  Generated: many green trees are in a piece of green trees
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many green trees are in a piece of green trees
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many green trees are in a piece of green trees
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is lying on the

Epoch [4/20], Average Training Loss: 2.7245, Current LR: 0.0002

--- Generating Example Captions ---

Example 1:
  Generated: many buildings and some green trees are in a dense residential area
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: many buildings and some green trees are in a commercial area
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many buildings and some green trees are in a dense residential area
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a p

...

Epoch [16/20], Average Training Loss: 2.0666, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are located in an industrial area
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: a playground is surrounded by many green trees and a road
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many buildings are around a playground with a parking lot
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on 

Epoch [17/20], Average Training Loss: 2.0515, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are located in an industrial area
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: a playground is surrounded by many green trees and a road
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many buildings are around a playground with a playground
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on a

Epoch [18/20], Average Training Loss: 2.0491, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are located in an industrial area
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: a playground is surrounded by many green trees and a road
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many buildings are around a playground with a playground
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on a

Epoch [19/20], Average Training Loss: 2.0499, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are in a commercial area
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: a playground is surrounded by many green trees and a road
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many buildings are around a playground with a playground
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connec

Epoch [20/20], Average Training Loss: 2.0568, Current LR: 2.0000000000000003e-06

--- Generating Example Captions ---

Example 1:
  Generated: many buildings are in a commercial area
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: a playground is surrounded by many green trees and a road
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many buildings are around a playground with a playground
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connec

In [49]:
torch.save(model.state_dict(), '/Users/arnavagarwal/Documents/Sem7/EE782/Assignment1/cnn_lstm_mobilenet.pth')
print("Model weights saved to cnn_lstm_mobilenet.pth")

Model weights saved to cnn_lstm_mobilenet.pth


In [ ]:

# Model Hyperparameters
ENCODER_DIM = 1280  
D_MODEL = 512      # The main dimension of the Transformer
NHEAD = 8          # Number of attention heads
NUM_LAYERS_TR = 4  # Number of decoder layers
VOCAB_SIZE = len(vocab)

# Training Hyperparameters
LEARNING_RATE_TR = 2e-4
NUM_EPOCHS_TR = 20 # Transformers can take a bit longer to converge
BATCH_SIZE = 64

# Setup
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA (NVIDIA GPU)")
# Check for MPS (Apple Silicon GPU)
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS (Apple Silicon GPU)")
# Fallback to CPU
else:
    device = torch.device("cpu")
    print("Using CPU")
print(f"Using device for Transformer: {device}")

# Instantiate the Model, Loss, and Optimizer
model_tr = EncoderDecoderTransformer(ENCODER_DIM, D_MODEL, NHEAD, NUM_LAYERS_TR, VOCAB_SIZE).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.stoi['<pad>'])
optimizer_tr = torch.optim.Adam(model_tr.parameters(), lr=LEARNING_RATE_TR)
scheduler_tr = torch.optim.lr_scheduler.StepLR(optimizer_tr, step_size=7, gamma=0.1)
# --- 3. Training and Evaluation Loops for Transformer ---

def create_causal_mask(size, device):
    return nn.Transformer.generate_square_subsequent_mask(size, device=device)

def create_padding_mask(seq, pad_idx, device):
    return (seq == pad_idx).to(device)

def train_one_epoch_transformer(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for features, captions, _ in tqdm(loader, desc="Training Transformer"):
        features, captions = features.to(device), captions.to(device)
        optimizer.zero_grad()
        
        tgt_in = captions[:, :-1]
        tgt_out = captions[:, 1:]
        
        tgt_mask = create_causal_mask(tgt_in.size(1), device)
        tgt_padding_mask = create_padding_mask(tgt_in, vocab.stoi['<pad>'], device)

        outputs = model(features, tgt_in, tgt_mask, tgt_padding_mask)
        loss = criterion(outputs.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(loader)

def evaluate_transformer(model, loader, vocab):
    model.eval()
    print("\n--- Generating Example Captions (Transformer) ---")
    with torch.no_grad():
        features, _, all_captions_strs = next(iter(loader))
        features = features.to(device)
        for i in range(min(5, len(features))):
            feature = features[i].unsqueeze(0)
            generated_caption_words = model.decoder.generate_caption(feature, vocab, max_len=MAX_LENGTH)
            generated_caption = ' '.join(generated_caption_words)
            print(f"\nExample {i+1}:")
            print(f"  Generated: {generated_caption}")
            gt_captions = all_captions_strs[i]
            print(f"  Ground Truth 1: {gt_captions[0]}")
            if len(gt_captions) > 1: print(f"  Ground Truth 2: {gt_captions[1]}")
    print("--- Evaluation Complete ---\n")

# --- 4. The Main Training Execution Block ---

print("\n--- Starting Transformer Model Training ---")
for epoch in range(1, NUM_EPOCHS_TR + 1):
    avg_train_loss = train_one_epoch_transformer(model_tr, train_loader, optimizer_tr, criterion)
    scheduler_tr.step()
    current_lr = optimizer_tr.param_groups[0]['lr']
    
    print(f"Epoch [{epoch}/{NUM_EPOCHS_TR}], Loss: {avg_train_loss:.4f}, LR: {current_lr}")
    evaluate_transformer(model_tr, val_loader, vocab)

print("Transformer training finished.")
torch.save(model_tr.state_dict(), 'cnn_transformer_baseline_mobilenet.pth')
print("Transformer model weights saved to cnn_transformer_baseline_mobilenet.pth")

Using MPS (Apple Silicon GPU)
Using device for Transformer: mps

--- Starting Transformer Model Training ---


Epoch [1/20], Loss: 3.0310, LR: 0.0002

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: many buildings are in a commercial area
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: a playground with a road is close to a road
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: many buildings are in a commercial area
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is lying on the bareland  near which there 

Epoch [2/20], Loss: 2.0059, LR: 0.0002

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near some buildings
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: some planes are in an airport near several buildings
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: some buildings and some green trees are in a commercial area
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is 

Epoch [3/20], Loss: 1.7916, LR: 0.0002

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are parked near a terminal in an airport
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: some planes are parked near a terminal at an airport
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: some planes are parked near a building in an airport
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected to a runway is lying

Epoch [4/20], Loss: 1.6550, LR: 0.0002

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are parked near a terminal in an airport
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: a building with a parking lot and a parking lot are close to a building
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: some planes are parked near a building in an airport
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and connected t

...

Epoch [16/20], Loss: 1.1630, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near several buildings
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: some planes are parked near several buildings in an airport
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: some planes are parked near several buildings in an airport
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and

Epoch [17/20], Loss: 1.1776, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near several buildings
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: several planes are parked near a terminal in an airport
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: some planes are parked near several buildings in an airport
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and con

Epoch [18/20], Loss: 1.1696, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near several buildings
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: some planes are parked near several buildings in an airport
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: some planes are parked near several buildings in an airport
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on and

Epoch [19/20], Loss: 1.1679, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are parked near several buildings in an airport
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: several planes are parked near a terminal in an airport
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: some planes are parked near several buildings in an airport
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on 

Epoch [20/20], Loss: 1.1724, LR: 2.0000000000000003e-06

--- Generating Example Captions (Transformer) ---

Example 1:
  Generated: some planes are in an airport near several buildings
  Ground Truth 1: the asphalted and airport runways divide the field into several rounded rectangles arranged next to which buildings and a road are located
  Ground Truth 2: the tarmac and airport runways divide the field into several orderly arranged rounded rectangles  next to which is buildings and a road

Example 2:
  Generated: several planes are parked in an airport near several buildings
  Ground Truth 1: Many white planes are parked at the airport
  Ground Truth 2: a motorway is built next to the airport

Example 3:
  Generated: some planes are parked near several buildings in an airport
  Ground Truth 1: a parking apron with an airplane parked on and connected to a runway is lying on bare ground near which there are some square buildings
  Ground Truth 2: a parking apron with a plane parked on 

Evaluation, Analysis & Explainability [2]

# Evaluation, Analysis & Explainability

In [51]:
# Run this cell once to install and set up the necessary libraries
!pip install torchmetrics
!pip install nltk
import nltk
nltk.download('wordnet')

  Using cached torch-2.8.0-cp313-none-macosx_11_0_arm64.whl.metadata (30 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 7.0 MB/s eta 0:00:00
Using cached torch-2.8.0-cp313-none-macosx_11_0_arm64.whl (73.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchmetrics] [torchmetrics]


[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/arnavagarwal/nltk_data...


True

## Metrics

In [ ]:
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score
from tqdm import tqdm
import numpy as np

ENCODER_DIM = 1280
def calculate_metrics(model, dataloader, vocab, device):
    """
    Calculates all required metrics (BLEU, METEOR, Length Stats, Repetitions)
    for a given model on a test dataset.
    """
    model.eval()
    
    # Store all ground truth captions and generated captions
    all_references = [] # A list of lists of lists of strings
    all_hypotheses = [] # A list of lists of strings
    
    # Store stats for length and repetition
    generated_lengths = []
    degenerate_count = 0
    total_count = 0

    with torch.no_grad():
        for features, _, all_captions_strs in tqdm(dataloader, desc="Calculating Metrics on Test Set"):
            features = features.to(device)
            
            for i in range(features.size(0)):
                feature = features[i].unsqueeze(0)
                
                if isinstance(model.decoder, LSTMDecoder):
                    generated_caption_words = model.decoder.generate_caption_beam_search(feature, vocab)
                elif isinstance(model.decoder, TransformerDecoderModel):
                    generated_caption_words = model.decoder.generate_caption(feature, vocab)
                else:
                    raise TypeError("Unknown decoder type")
                
                # --- Store results for BLEU/METEOR ---
                # NLTK expects a list of reference translations for each hypothesis
                # all_captions_strs[i] is already a list of strings
                # We need to tokenize them
                references = [vocab.tokenizer(ref) for ref in all_captions_strs[i]]
                
                all_references.append(references)
                all_hypotheses.append(generated_caption_words)
                
                # --- Calculate Length and Repetition Stats ---
                generated_lengths.append(len(generated_caption_words))
                total_count += 1
                
                # Check for degenerate repetitions (3 or more identical tokens in a row)
                if len(generated_caption_words) >= 3:
                    for j in range(len(generated_caption_words) - 2):
                        if generated_caption_words[j] == generated_caption_words[j+1] == generated_caption_words[j+2]:
                            degenerate_count += 1
                            break # Only count once per caption


   
    bleu1 = corpus_bleu(all_references, all_hypotheses, weights=(1, 0, 0, 0))
    bleu2 = corpus_bleu(all_references, all_hypotheses, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(all_references, all_hypotheses, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(all_references, all_hypotheses, weights=(0.25, 0.25, 0.25, 0.25))


    meteor_scores = [meteor_score(refs, hyp) for refs, hyp in zip(all_references, all_hypotheses)]
    avg_meteor = np.mean(meteor_scores)

  
    avg_len = np.mean(generated_lengths)
    std_len = np.std(generated_lengths)

    repetition_percent = (degenerate_count / total_count) * 100 if total_count > 0 else 0
    
 
    metrics = {
        "BLEU-1": bleu1,
        "BLEU-2": bleu2,
        "BLEU-3": bleu3,
        "BLEU-4": bleu4,
        "METEOR": avg_meteor,
        "Avg. Length": f"{avg_len:.2f} ± {std_len:.2f}",
        "Repetition % (>=3)": f"{repetition_percent:.2f}%"
    }
    
    print("\n--- Final Metrics ---")
    for name, value in metrics.items():
        if isinstance(value, float):
            print(f"{name:<20} {value:.4f}")
        else:
            print(f"{name:<20} {value}")
            
    return metrics

# --- How to Use This Script ---


test_feature_dir = './test_resnet_features' # Or './test_mobilenet_v2_features'
test_dataset = CaptionDataset(test_df, vocab, test_feature_dir, MAX_LENGTH)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

model_to_evaluate = EncoderDecoder(ENCODER_DIM, EMBED_DIM, HIDDEN_DIM, VOCAB_SIZE, NUM_LAYERS).to(device)
model_to_evaluate.load_state_dict(torch.load('cnn_lstm_resnet.pth', map_location=device))


print("Calculating metrics for the CNN+LSTM (ResNet-18) model...")
lstm_metrics = calculate_metrics(model_to_evaluate, test_loader, vocab, device)




model_tr_to_evaluate = EncoderDecoderTransformer(ENCODER_DIM, D_MODEL, NHEAD, NUM_LAYERS_TR, VOCAB_SIZE).to(device)
model_tr_to_evaluate.load_state_dict(torch.load('cnn_transformer_baseline_resnet.pth', map_location=device))

print("\n\nCalculating metrics for the CNN+Transformer (ResNet-18) model...")
transformer_metrics = calculate_metrics(model_tr_to_evaluate, test_loader, vocab, device)



Calculating metrics for the CNN+LSTM (ResNet-18) model...



--- Final Metrics ---
BLEU-1               0.5343
BLEU-2               0.3420
BLEU-3               0.2401
BLEU-4               0.1746
METEOR               0.3692
Avg. Length          9.98 ± 2.42
Repetition % (>=3)   0.00%


Calculating metrics for the CNN+Transformer (ResNet-18) model...



--- Final Metrics ---
BLEU-1               0.6286
BLEU-2               0.4334
BLEU-3               0.3194
BLEU-4               0.2394
METEOR               0.4523
Avg. Length          10.46 ± 2.56
Repetition % (>=3)   0.00%


In [ ]:
from nltk.translate.bleu_score import corpus_bleu
from nltk.translate.meteor_score import meteor_score
from tqdm import tqdm
import numpy as np

ENCODER_DIM = 1280
def calculate_metrics(model, dataloader, vocab, device):
    """
    Calculates all required metrics (BLEU, METEOR, Length Stats, Repetitions)
    for a given model on a test dataset.
    """
    model.eval()
    
    # Store all ground truth captions and generated captions
    all_references = [] # A list of lists of lists of strings
    all_hypotheses = [] # A list of lists of strings
    
    # Store stats for length and repetition
    generated_lengths = []
    degenerate_count = 0
    total_count = 0

    with torch.no_grad():
        for features, _, all_captions_strs in tqdm(dataloader, desc="Calculating Metrics on Test Set"):
            features = features.to(device)
            
            for i in range(features.size(0)):
                feature = features[i].unsqueeze(0)
                
                if isinstance(model.decoder, LSTMDecoder):
                    # Using beam search for the final LSTM evaluation
                    generated_caption_words = model.decoder.generate_caption_beam_search(feature, vocab)
                elif isinstance(model.decoder, TransformerDecoderModel):
                    generated_caption_words = model.decoder.generate_caption(feature, vocab)
                else:
                    raise TypeError("Unknown decoder type")
                
                references = [vocab.tokenizer(ref) for ref in all_captions_strs[i]]
                
                all_references.append(references)
                all_hypotheses.append(generated_caption_words)
                
                # --- Calculate Length and Repetition Stats ---
                generated_lengths.append(len(generated_caption_words))
                total_count += 1
                
                # Check for degenerate repetitions (3 or more identical tokens in a row)
                if len(generated_caption_words) >= 3:
                    for j in range(len(generated_caption_words) - 2):
                        if generated_caption_words[j] == generated_caption_words[j+1] == generated_caption_words[j+2]:
                            degenerate_count += 1
                            break # Only count once per caption

    bleu1 = corpus_bleu(all_references, all_hypotheses, weights=(1, 0, 0, 0))
    bleu2 = corpus_bleu(all_references, all_hypotheses, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(all_references, all_hypotheses, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(all_references, all_hypotheses, weights=(0.25, 0.25, 0.25, 0.25))

    meteor_scores = [meteor_score(refs, hyp) for refs, hyp in zip(all_references, all_hypotheses)]
    avg_meteor = np.mean(meteor_scores)


    avg_len = np.mean(generated_lengths)
    std_len = np.std(generated_lengths)


    repetition_percent = (degenerate_count / total_count) * 100 if total_count > 0 else 0
    

    metrics = {
        "BLEU-1": bleu1,
        "BLEU-2": bleu2,
        "BLEU-3": bleu3,
        "BLEU-4": bleu4,
        "METEOR": avg_meteor,
        "Avg. Length": f"{avg_len:.2f} ± {std_len:.2f}",
        "Repetition % (>=3)": f"{repetition_percent:.2f}%"
    }
    
    print("\n--- Final Metrics ---")
    for name, value in metrics.items():
        if isinstance(value, float):
            print(f"{name:<20} {value:.4f}")
        else:
            print(f"{name:<20} {value}")
            
    return metrics

test_feature_dir = './test_mobilenet_features' # Or './test_mobilenet_v2_features'
test_dataset = CaptionDataset(test_df, vocab, test_feature_dir, MAX_LENGTH)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

model_to_evaluate = EncoderDecoder(ENCODER_DIM, EMBED_DIM, HIDDEN_DIM, VOCAB_SIZE, NUM_LAYERS).to(device)
model_to_evaluate.load_state_dict(torch.load('cnn_lstm_mobilenet.pth', map_location=device))
print("Calculating metrics for the CNN+LSTM (MobileNet) model...")
lstm_metrics = calculate_metrics(model_to_evaluate, test_loader, vocab, device)

model_tr_to_evaluate = EncoderDecoderTransformer(ENCODER_DIM, D_MODEL, NHEAD, NUM_LAYERS_TR, VOCAB_SIZE).to(device)
model_tr_to_evaluate.load_state_dict(torch.load('cnn_transformer_baseline_mobilenet.pth', map_location=device))

print("\n\nCalculating metrics for the CNN+Transformer (MobileNet) model...")
transformer_metrics = calculate_metrics(model_tr_to_evaluate, test_loader, vocab, device)

Calculating metrics for the CNN+LSTM (MobileNet) model...



--- Final Metrics ---
BLEU-1               0.5424
BLEU-2               0.3509
BLEU-3               0.2445
BLEU-4               0.1765
METEOR               0.3783
Avg. Length          9.80 ± 2.40
Repetition % (>=3)   0.00%


Calculating metrics for the CNN+Transformer (MobileNet) model...



--- Final Metrics ---
BLEU-1               0.6372
BLEU-2               0.4463
BLEU-3               0.3321
BLEU-4               0.2513
METEOR               0.4604
Avg. Length          10.43 ± 2.65
Repetition % (>=3)   0.00%


#### Clearly among CNN(ResNet-18) + LSTM, CNN(MobileNetV2) + LSTM, CNN(ResNet-18) + Transformer, (Optional) CNN(MobileNetV2) + Transformer **The best one performing is CNN + Transformer (MobileNet)**

In [80]:
pip install grad-cam


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [71]:
pip install opencv-python


  Using cached opencv_python-4.12.0.88-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
Using cached opencv_python-4.12.0.88-cp37-abi3-macosx_13_0_arm64.whl (37.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 10.8 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.2
    Uninstalling numpy-2.3.2:
      Successfully uninstalled numpy-2.3.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [opencv-python]0m [opencv-python]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Qualitative & Slice Analysis

In [ ]:
success_filenames = [
    'farmland_370.jpg', 'medium_residential_68.jpg', 'river_401.jpg', 'baseballfield_52.jpg', 'airport_360.jpg', 'bareland_49.jpg', 'beach_42.jpg', 'baseballfield_69.jpg', 'farmland_58.jpg', 'river_50.jpg'
]
failure_filenames = [
    'airport_40.jpg', 'railwaystation_62.jpg', 'mountain_50.jpg', 'airport_355.jpg', 'park_38.jpg',
    'forest_58.jpg', 'stadium_48.jpg', 'industrial_45.jpg', '00656.jpg', 'park_67.jpg'
]


class FullEncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super(FullEncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

TRANSFORMER_ENCODER_DIM = 1280 
D_MODEL = 512
NHEAD = 8
NUM_LAYERS_TR = 4
VOCAB_SIZE = len(vocab)


trained_model_wrapper_tr = EncoderDecoderTransformer(
    TRANSFORMER_ENCODER_DIM, D_MODEL, NHEAD, NUM_LAYERS_TR, VOCAB_SIZE
).to(device)

trained_model_wrapper_tr.load_state_dict(torch.load('cnn_transformer_baseline_mobilenet.pth', map_location=device))
print("Successfully loaded trained model weights.")

trained_decoder = trained_model_wrapper_tr.decoder

cnn_encoder = CNNEncoder(model_name='mobilenet_v2').to(device)

full_model = FullEncoderDecoder(cnn_encoder, trained_decoder)
full_model.eval()

def display_examples(filenames, df, model, transform, vocab, device, target_layer_type, title):
    print(f"\n--- Displaying {title} Examples ---")
    for filename in filenames:
        # Find row by filename (partial match for flexibility)
        row = df[df['filename'].str.contains(filename, na=False)]
        if row.empty:
            print(f"Image '{filename}' not found in DataFrame.")
            continue
        df_row = row.iloc[0]
        print(f"\nImage: {df_row['filename']}")
        print(f"Ground Truth Captions: {df_row['captions']}")
        try:
            # Generate caption
            image_bytes_str = df_row['image']
            image_dict = ast.literal_eval(image_bytes_str)
            image_bytes = image_dict['bytes']
            img_pil = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            input_tensor = transform(img_pil).unsqueeze(0).to(device)
            with torch.no_grad():
                features = model.encoder(input_tensor)
                if hasattr(model.decoder, 'generate_caption_beam_search'):
                    generated_caption_words = model.decoder.generate_caption_beam_search(features, vocab)
                else:
                    generated_caption_words = model.decoder.generate_caption(features, vocab)
            generated_caption = ' '.join(generated_caption_words)
            print(f"Generated Caption: {generated_caption}")
            # Display image
            plt.imshow(img_pil)
            plt.axis('off')
            plt.title(f"{title}: {filename}")
            plt.show()
        except Exception as e:
            print(f"Error processing '{filename}': {e}")

# Display 10 successes
display_examples(success_filenames, test_df, full_model, image_transforms, vocab, device, nn.BatchNorm2d, "Success")

# Display 10 failures
display_examples(failure_filenames, test_df, full_model, image_transforms, vocab, device, nn.BatchNorm2d, "Failure")

Successfully loaded trained model weights.

--- Displaying Success Examples ---

Image: rsicd_images/farmland_370.jpg
Ground Truth Captions: ['the green and creamy agricultural land looks like a checker', 'the agricultural land that is green and creamy looks like a table of ladies', 'the farmland which is green and cream colored looks like a checkerboard', 'this farmland looks like mosaic with green squares and yellow ones', 'many pieces of green farmlands are together']
Generated Caption: many pieces of agricultural land are together


<Figure size 640x480 with 1 Axes>

Image 'medium_residential_68.jpg' not found in DataFrame.

Image: rsicd_images/river_401.jpg
Ground Truth Captions: ['pieces of agricultural land are found on the banks of the rapidly flowing river', 'there is a river passing by the farmland and towns', 'pieces of farmlands are at the bank of the fast flowing river', 'this river passes through those blocks of farmland on its bank', 'urban areas and many pieces of green farmlands are in two sides of a curved river']
Generated Caption: many green trees and some buildings are located on two sides of a curved river


<Figure size 640x480 with 1 Axes>


Image: rsicd_images/baseballfield_52.jpg
Ground Truth Captions: ["There's a baseball field around the baseball field", 'some large trees were planted around the baseball field', 'some tall trees were planted around the baseball field', "There's a baseball field around the baseball field", 'there is a baseball field around the baseball field']
Generated Caption: a baseball field is surrounded by some green trees


...

<Figure size 640x480 with 1 Axes>


Image: rsicd_images/00656.jpg
Ground Truth Captions: ['a half-old is situated near the distictive buildings', 'a playground is near three basketball fields and several buildings', 'a playground is next to several large buildings', 'a playground is next to several buildings', 'a playground is next to several large buildings']
Generated Caption: a playground is surrounded by some green trees and many buildings


<Figure size 640x480 with 1 Axes>


Image: rsicd_images/park_67.jpg
Ground Truth Captions: ['the park with different swimming pools is next to a parking lot', 'there are three swimming pools and some buildings between a lawn and a parking lot', 'the water park with many pools is next to a parking lot', 'two swimming pools are located in this park', 'a large buildings with a parking lot is near a park with some green plants']
Generated Caption: many green trees and a large pond are in a park near a road


<Figure size 640x480 with 1 Axes>

### 3 error slices

In [84]:
# --- Create Slice 1: rails scenes ---
rails_df = test_df[test_df['captions'].apply(lambda captions: any('rails' in c for c in captions))]
print(f"Found {len(rails_df)} images for the 'rails' slice.")

# --- Create Slice 2: rivers, swimming pools captions ---
rivers_df = test_df[test_df['captions'].apply(lambda captions: any('rivers' in c for c in captions))]
print(f"Found {len(rivers_df)} images for the 'rivers' slice.")

# --- Create Slice 3: cars scenes ---
cars_df = test_df[test_df['captions'].apply(lambda captions: any('cars' in c for c in captions))]
print(f"Found {len(cars_df)} images for the 'cars' slice.")


Found 23 images for the 'rails' slice.
Found 21 images for the 'rivers' slice.
Found 134 images for the 'cars' slice.


### BLEU-4 deltas

In [ ]:
def get_bleu4_on_slice(df, model, vocab, device):
    if len(df) == 0: return 0.0
    feature_dir = './test_mobilenet_features'
    dataset = CaptionDataset(df, vocab, feature_dir, MAX_LENGTH)
    loader = DataLoader(dataset, batch_size=32, collate_fn=collate_fn)
    metrics = calculate_metrics(model, loader, vocab, device)
    return metrics['BLEU-4']

bleu4_rails = get_bleu4_on_slice(rails_df, model_to_evaluate, vocab, device)
bleu4_rivers = get_bleu4_on_slice(rivers_df, model_to_evaluate, vocab, device)
bleu4_cars = get_bleu4_on_slice(cars_df, model_to_evaluate, vocab, device)


overall_bleu4 = 0.2513
delta_rails = bleu4_rails - overall_bleu4
delta_rivers = bleu4_rivers - overall_bleu4
delta_cars = bleu4_cars - overall_bleu4



--- Final Metrics ---
BLEU-1               0.4980
BLEU-2               0.2969
BLEU-3               0.2054
BLEU-4               0.1499
METEOR               0.3232
Avg. Length          10.78 ± 2.26
Repetition % (>=3)   0.00%



--- Final Metrics ---
BLEU-1               0.5028
BLEU-2               0.3004
BLEU-3               0.2034
BLEU-4               0.1205
METEOR               0.3550
Avg. Length          8.43 ± 1.92
Repetition % (>=3)   0.00%



--- Final Metrics ---
BLEU-1               0.5736
BLEU-2               0.3866
BLEU-3               0.2781
BLEU-4               0.2071
METEOR               0.4300
Avg. Length          10.00 ± 2.46
Repetition % (>=3)   0.00%


### Plot per-slice BLEU-4 deltas

In [86]:
slice_names = ['Rails', 'Rivers', 'Cars']
deltas = [delta_rails, delta_rivers, delta_cars]

plt.figure(figsize=(8, 5))
plt.bar(slice_names, deltas, color=['red', 'orange', 'blue'])
plt.ylabel('BLEU-4 Delta (vs. Overall)')
plt.title('Performance Drop on Challenging Slices')
plt.axhline(0, color='grey', linestyle='--')
plt.show()

<Figure size 800x500 with 1 Axes>

## Grad-CAM

In [102]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import numpy as np
import cv2 # OpenCV for image manipulation
import matplotlib.pyplot as plt
import random
import ast # To parse the string representation of the dictionary
import io  # To handle bytes in memory

# Make sure you have this installed: !pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image


# --- HELPER FUNCTION FOR VISUALIZATION (Corrected for Byte-based Loading) ---
def visualize_and_generate_from_row(model, df_row, transform, vocab, device, target_layer_type):
    """
    Generates a caption and Grad-CAM heatmap from a DataFrame row containing image bytes.
    """
    # 1. Load and process the image FROM THE DATAFRAME ROW
    image_bytes_str = df_row['image']
    image_dict = ast.literal_eval(image_bytes_str)
    image_bytes = image_dict['bytes']
    img_pil = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    
    input_tensor = transform(img_pil).unsqueeze(0).to(device)
    
    # 2. Get the CNN encoder from the model
    encoder = model.encoder.cnn
    encoder.eval()

    # 3. Set up Grad-CAM
    target_layers = [module for module in encoder.modules() if isinstance(module, target_layer_type)][-1:]
    if not target_layers:
        raise ValueError(f"Could not find any layers of type {target_layer_type} in the encoder.")

    class FeatureVectorTarget:
        def __call__(self, model_output):
            return model_output.sum()
            
    cam = GradCAM(model=encoder, target_layers=target_layers)
    targets = [FeatureVectorTarget()]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

    # 4. Generate the caption
    model.eval()
    with torch.no_grad():
        features = model.encoder(input_tensor)
        if isinstance(model.decoder, LSTMDecoder):
            generated_caption_words = model.decoder.generate_caption_beam_search(features, vocab)
        else: # Transformer
            generated_caption_words = model.decoder.generate_caption(features, vocab)
    generated_caption = ' '.join(generated_caption_words)

    # 5. Overlay heatmap on the original image
    img_np = np.array(img_pil.resize((224, 224))) / 255.0
    visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
    
    # 6. Display everything
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))
    # Use the filename from the row for context
    original_filename = df_row['filename']
    fig.suptitle(f"Generated Caption: {' '.join(generated_caption.split()[:15])}...", fontsize=14, wrap=True)
    

    axs[0].imshow(img_pil)
    axs[0].set_title(f"Original Image\n({original_filename})")
    axs[0].axis('off')
    
    axs[1].imshow(visualization)
    axs[1].set_title("Grad-CAM Heatmap")
    axs[1].axis('off')
    
    plt.tight_layout()
    plt.show()


# --- STEP 1: DEFINE THE FullEncoderDecoder WRAPPER CLASS ---
class FullEncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super(FullEncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder


# --- STEP 2: SET UP AND RUN THE VISUALIZATION ---

# Define architecture parameters for the model you are loading (e.g., ResNet-LSTM)
TRANSFORMER_ENCODER_DIM = 1280 # MobileNet output
D_MODEL = 512
NHEAD = 8
NUM_LAYERS_TR = 4 # CRITICAL: Must match the NUM_LAYERS_TR you trained with
VOCAB_SIZE = len(vocab)

# Create an instance of the model you trained (the EncoderDecoder wrapper)
trained_model_wrapper_tr = EncoderDecoderTransformer(
    TRANSFORMER_ENCODER_DIM, D_MODEL, NHEAD, NUM_LAYERS_TR, VOCAB_SIZE
).to(device)

# Load the state dict into this wrapper model
trained_model_wrapper_tr.load_state_dict(torch.load('cnn_transformer_baseline_mobilenet.pth', map_location=device))
print("Successfully loaded trained model weights.")

# Extract the trained decoder part
trained_decoder = trained_model_wrapper_tr.decoder

# Create the CNN encoder needed for Grad-CAM
cnn_encoder = CNNEncoder(model_name='mobilenet_v2').to(device)

# Assemble the final model for analysis
full_model = FullEncoderDecoder(cnn_encoder, trained_decoder)
full_model.eval()

# --- RUN THE VISUALIZATION (Corrected Loop) ---
print("\n--- Generating examples with Grad-CAM from DataFrame ---")
# for _ in range(30): # Generate 15 examples to get a good sample
#     # Randomly sample one row from your test DataFrame
#     random_row = test_df.sample(1).iloc[0]
    
#     print(f"\nAnalyzing image: {random_row['filename']}")
#     try:
#         # Pass the entire row to the visualization function
#         visualize_and_generate_from_row(
#             full_model, 
#             random_row, 
#             image_transforms, 
#             vocab, 
#             device, 
#             target_layer_type=nn.BatchNorm2d
#         )
#         print("Original Caption:", random_row['captions'][0])
#     except Exception as e:
#         print(f"  > An error occurred: {e}")

def analyze_image_by_filename(filename, df, model, transform, vocab, device, target_layer):
    """
    Finds an image by its filename in the DataFrame and runs the visualization.
    
    Args:
        filename (str): The base filename of the image (e.g., 'airport_10.jpg').
        df (pd.DataFrame): The DataFrame to search in (e.g., test_df).
        ... (rest of the arguments are for the visualization function)
    """
    target_row = df[df['filename'].str.contains(filename, na=False)]
    
    if target_row.empty:
        print(f"Error: Could not find any image with filename containing '{filename}' in the DataFrame.")
        return
        
    row_data = target_row.iloc[0]
    
    print(f"--- Analyzing Specific Image: {row_data['filename']} ---")
    try:
        # Call our existing visualization function with the found row
        visualize_and_generate_from_row(
            model, 
            row_data, 
            transform, 
            vocab, 
            device, 
            target_layer
        )
        print("Original Caption:", row_data['captions'][0])
    except Exception as e:
        print(f"  > An error occurred during visualization: {e}")

filename_to_analyze_list_good = [
    'farmland_370.jpg', 'river_401.jpg', 'baseballfield_52.jpg', 'bareland_49.jpg', 'church_60.jpg',
]
filename_to_analyze_list_bad = [
    'railwaystation_62.jpg', 'mountain_50.jpg', 'airport_355.jpg', 'stadium_48', 'park_38.jpg',
]

# Let's modify the below code to first analyze the good ones and then the bad ones
print("\n--- Analyzing 'Good' Examples ---")
for filename in filename_to_analyze_list_good:
    analyze_image_by_filename(
        filename,
        test_df,
        full_model,
        image_transforms,
        vocab,
        device,
        target_layer=nn.BatchNorm2d
    )

print("\n--- Analyzing 'Bad' Examples ---")
for filename in filename_to_analyze_list_bad:
    analyze_image_by_filename(
        filename,
        test_df,
        full_model,
        image_transforms,
        vocab,
        device,
        target_layer=nn.BatchNorm2d
    )
    

Successfully loaded trained model weights.

--- Generating examples with Grad-CAM from DataFrame ---

--- Analyzing 'Good' Examples ---
--- Analyzing Specific Image: rsicd_images/farmland_370.jpg ---


<Figure size 1200x600 with 2 Axes>

Original Caption: the green and creamy agricultural land looks like a checker
--- Analyzing Specific Image: rsicd_images/river_401.jpg ---


<Figure size 1200x600 with 2 Axes>

Original Caption: pieces of agricultural land are found on the banks of the rapidly flowing river
--- Analyzing Specific Image: rsicd_images/baseballfield_52.jpg ---


...

Original Caption: Many white planes are parked at the airport
--- Analyzing Specific Image: rsicd_images/stadium_48.jpg ---


<Figure size 1200x600 with 2 Axes>

Original Caption: There's a big stadium next to the road
--- Analyzing Specific Image: rsicd_images/park_38.jpg ---


<Figure size 1200x600 with 2 Axes>

Original Caption: A beautiful park is built next to the road


## Mis-caption case study

### Example 1:

In [ ]:
def analyze_image_by_filename(filename, df, model, transform, vocab, device, target_layer):
    """
    Finds an image by its filename in the DataFrame and runs the visualization.
    
    Args:
        filename (str): The base filename of the image (e.g., 'airport_10.jpg').
        df (pd.DataFrame): The DataFrame to search in (e.g., test_df).
        ... (rest of the arguments are for the visualization function)
    """
    target_row = df[df['filename'].str.contains(filename, na=False)]
    
    if target_row.empty:
        print(f"Error: Could not find any image with filename containing '{filename}' in the DataFrame.")
        return
        
    row_data = target_row.iloc[0]
    
    print(f"--- Analyzing Specific Image: {row_data['filename']} ---")
    try:
        # Call our existing visualization function with the found row
        visualize_and_generate_from_row(
            model, 
            row_data, 
            transform, 
            vocab, 
            device, 
            target_layer
        )
        print("Original Caption:", row_data['captions'][0])
    except Exception as e:
        print(f"  > An error occurred during visualization: {e}")

filename_to_analyze = 'airport_355.jpg'

analyze_image_by_filename(
    filename_to_analyze,
    test_df, 
    full_model, 
    image_transforms,
    vocab,
    device,
    target_layer=nn.BatchNorm2d # The target layer for Grad-CAM
)


--- Analyzing Specific Image: rsicd_images/airport_355.jpg ---


<Figure size 1200x600 with 2 Axes>

Original Caption: Many white planes are parked at the airport


##### Image: An image clearly showing an airport with a airplanes.
##### Generated Caption: "a large number of cars parked near the airport plains"
##### Ground Truth: "Many white planes are parked at the airport"

--- 

#### Hypothesizing the Cause:
##### Tiny Objects: "The model failed to identify the airplane. My hypothesis is that the global average pooling in the Mobilenet_v2 encoder 'blurs out' features from very small objects, causing the signal for 'airplane' to be lost. The remaining features (paved ground, lines) are more similar to a parking lot."

--- 

#### Propose a Non-Advanced Fix
##### Fix for Tiny Objects: "A potential fix would be to use an encoder with an attention mechanism (like a Vision Transformer) or to avoid global average pooling and instead use a spatial attention mechanism over the final feature map. This would allow the decoder to focus on small regions of the image instead of just the global summary."

---

### Example 2

In [ ]:
filename_to_analyze_2 = 'railwaystation_62.jpg'
analyze_image_by_filename(
    filename_to_analyze_2,
    test_df,
    full_model,
    image_transforms,
    vocab,
    device,
    target_layer=nn.BatchNorm2d
)

--- Analyzing Specific Image: rsicd_images/railwaystation_62.jpg ---


<Figure size 1200x600 with 2 Axes>

Original Caption: the pin-shaped rails covered with four red-banded ceilings are surrounded by houses


##### Image: An image clearly showing a set of railway lines, possibly in a residential or industrial area, with some unique red-roofed structures over the tracks.
##### Generated Caption: "many buildings and some green trees are in a commercial area"
##### Ground Truth: "the pin-shaped rails covered with four red-banded ceilings are surrounded by houses"

--- 

#### Hypothesizing the Cause:
##### Primary Hypothesis (Vocabulary Gap & Feature Confusion): The model likely has a very weak understanding of "railway" or "tracks." These words may be rare in the training data, leading to a poorly learned or non-existent representation in the vocabulary (Vocabulary Gap). When the CNN encoder processes the image, it extracts features representing long, parallel lines and surrounding structures. Because the decoder's strongest and most confident concepts are "roads," "buildings," and "trees," it maps these ambiguous line features to the closest common concept it knows: a "commercial area" (which often contains roads and buildings). The unique features of a railway are not distinct enough to overcome the model's strong bias towards more frequent classes.
##### Secondary Hypothesis (Domain Shift / Lack of Specificity): The ground truth caption is incredibly specific ("pin-shaped rails," "red-banded ceilings"). The training data likely contains many more generic captions. The model has learned that generating broad, general descriptions (like "commercial area") is a "safer" strategy that results in a lower average loss than attempting to generate highly specific, detailed descriptions which have a higher chance of being incorrect. It has optimized for generality over specificity.

--- 

#### Propose a Non-Advanced Fix
#### Fix for Vocabulary Gap (Data Augmentation): The most direct fix would be to augment the training data. We could gather more images containing railways, tracks, and train stations and add them to the training set. It would also be crucial to ensure these new images have captions that explicitly and repeatedly use words like "railway," "tracks," "train," and "station." This would strengthen the association between the visual features of rails and the corresponding tokens in the vocabulary, making it much less likely for the model to default to "road" or "commercial area."
---

### Example 3

In [ ]:
filename_to_analyze_2 = 'mountain_50.jpg'
analyze_image_by_filename(
    filename_to_analyze_2,
    test_df,
    full_model,
    image_transforms,
    vocab,
    device,
    target_layer=nn.BatchNorm2d
)

--- Analyzing Specific Image: rsicd_images/mountain_50.jpg ---


<Figure size 1200x600 with 2 Axes>

Original Caption: the snowy mountain is green and brown


#### Image: A satellite or aerial image of a mountain range with significant snow cover
#### Generated Caption: "many white waves are in a piece of green ocean"
#### Ground Truth: "the snowy mountain is green and brown"

--- 

### Hypothesizing the Cause:
#### Primary Hypothesis (Texture/Color Bias & Lack of Global Context): This is a classic case of the CNN encoder being biased by low-level features. The visual patterns of white snow drifting over green and brown terrain can create high-frequency, undulating textures. From a purely textural standpoint, these patterns are very similar to the appearance of white sea foam and waves on green or blue water. The ResNet-18 encoder, pre-trained on ground-level ImageNet photos, may not have a strong concept of "what a mountain looks like from above." Therefore, its feature vector likely encodes "white wavy patterns" and "green patches" very strongly. When this feature vector is passed to the decoder, the decoder finds that the most common co-occurrence of these features in its training data corresponds to the "ocean/waves" domain, and confidently generates that caption. It fails to use the global context (e.g., elevation, shadows) that would signify "mountain."

--- 

### Propose a Non-Advanced Fix
#### Fix for Texture/Color Bias (Data Augmentation): To break this false correlation, we need to show the model more counter-examples. The most effective fix would be to augment the dataset with more images of snowy mountains. Crucially, these new images should be paired with captions that explicitly use words like "mountain," "snow," "peak," and "ridge." This would force the model to learn that the "white wavy texture" can also map to the "mountain" domain, making it less likely to default to "ocean." Additionally, one could add color jitter (transforms.ColorJitter) to the image augmentation pipeline. By slightly altering the saturation and brightness, we can train the model to be less reliant on specific shades of green or white and focus more on larger contextual shapes and structures.